# Imports

In [1]:
import os
import requests
import pandas as pd
import talib
from datetime import date
from io import BytesIO
from time import time
from zipfile import ZipFile, BadZipFile

from dateutil.relativedelta import relativedelta

LAST_DATA_POINT_DELAY = 0
ITV_ALIASES = {"1m": "1min", "3m": "3min", "5m": "5min", "15m": "15min", "30m": "30min"}
DATA_DIR = 'data/'

# FUTURES_PAIRS = ['BTCUSDT', 'ETHUSDT', 'XRPUSDC', 'BNBUSDT', 'SOLUSDT', '']
PAIRS = ['SOLUSDT','DOGEUSDT', '1000PEPEUSDT', 'BTCUSDC', 'AVAXUSDC', 'BNTUSDT', '1000BONKUSDT', 'ADAUSDT', 'LINKUSDT', 'ARBUSDT']
INTERVALS = ['1m', '15m', '1h', '1d']
FUNCTIONS = ['ADX', 'APO', 'AROONOSC', 'BOP', 'CCI', 'CMO', 'MFI', 'MOM', 'ROCP', 'RSI', 'STOCHF', 'STOCHRSI', 'ULTOSC', 'WILLR', 'ADOSC', 'NATR']
MIN_DATE = "2024-03-01"

In [2]:
# all_columns_names = [
#     f"{p}{i}_{t}"
#     for p in PAIRS
#     for i in INTERVALS
#     for t in FUNCTIONS
# ]
# all_columns_names

# Define functions

In [3]:
def _fix_and_fill_df(df, itv):
    # print(f'df before duplicates drop {df}')
    df.drop_duplicates(inplace=True)
    # print(f'df after duplicates drop {df}')
    ### We need to delete faulty rows with string values as DataClient seems to write it sometimes
    drop_values = ["Opened", "Open", "High", "Low", "Close", "Volume"]
    df = df[~df.isin(drop_values).any(axis=1)]
    ### data_range does use 'T' instead of 'm' to mark minute freq
    freq = itv if "m" not in itv else ITV_ALIASES[itv]
    fixed_dates_df = pd.DataFrame(
        pd.date_range(start=df.iloc[0, 0], end=df.iloc[-1, 0], freq=freq),
        columns=["Opened"],
    )
    # print(f'fixed_dates_df {fixed_dates_df}')
    df["Opened"] = pd.to_datetime(df["Opened"], format="%Y-%m-%d %H:%M:%S")
    if len(fixed_dates_df) > len(df):
        df = fixed_dates_df.merge(df, on="Opened", how="left")
        df.ffill(inplace=True)
    ### Replacing 0 volume with the smallest one to make some TAcalculations possible
    col = df.columns[-1] # Assume that last column is Volume
    df[col] = df[col].astype('float64').replace(0.0, 1e-8)
    return df


def _download_and_unzip(url, output_path):
    try:
        response = requests.get(url)
        with ZipFile(BytesIO(response.content)) as zip_file:
            zip_file.extractall(output_path)
        return True
    except BadZipFile:
        return False


def _read_partial_df(_path):
    file_path = os.path.join(_path, os.listdir(_path)[0])
    # print(f'file_path {file_path}')
    df_temp = pd.read_csv(file_path, sep=",", usecols=[0, 1, 2, 3, 4, 5])
    df_temp.columns = [
        "Opened",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
    ]  # Nadajemy nazwy kolumn
    try:
        # Attempt to convert assuming microseconds
        df_temp["Opened"] = pd.to_datetime(df_temp["Opened"], unit="ms")
    except pd.errors.OutOfBoundsDatetime:
        # Handle the case where conversion fails
        print("Error: Timestamp conversion resulted in out-of-bounds datetime.")
        # You can choose to handle it differently, e.g., set to NaT or raise a custom error
        df_temp["Opened"] = pd.to_datetime(
            df_temp["Opened"], unit="us", errors="coerce"
        )
        if df_temp["Opened"].isna().any():
            raise ValueError(
                "Some timestamps could not be converted and are set to NaT."
            )

    return df_temp


def _collect_to_date(
        url, output_folder, start_date=date(year=2017, month=1, day=1), delta_itv="months"
):
    if delta_itv == "months":
        delta = relativedelta(months=1)
        end_date = date.today() - delta
        print(f"Collecting monthly from {start_date} to {end_date}")
    elif delta_itv == "days":
        delta = relativedelta(days=1)
        end_date = date.today() - delta
        print(f"Collecting daily from {start_date} to {end_date}")
    else:
        raise ValueError("arg delta_itv should be one of 'months' or 'days'")
    data_frames = []
    while start_date <= end_date:
        if delta_itv == "months":
            _url = url + f"{str(end_date)[:-3]}.zip"
            output_path = os.path.join(output_folder, f"{str(end_date)[:-3]}")
        elif delta_itv == "days":
            _url = url + f"{end_date}.zip"
            output_path = os.path.join(output_folder, f"{end_date}")
        if os.path.exists(output_path) and os.listdir(output_path)[0].endswith(".csv"):
            data_frames.append(_read_partial_df(output_path))
        else:
            print(f"downloading... {_url}")
            if _download_and_unzip(_url, output_path):
                data_frames.append(_read_partial_df(output_path))
            else:
                print(
                    f'"File is not a zip file" - {end_date} datapoint does not exist at BinanceVision'
                )
                # break
        end_date -= delta
    # Collecting starts from current date and ends in last existing datapoint,
    # so we need to reverse df order
    data_frames.reverse()
    return data_frames


def by_BinanceVision(
    ticker="BTCBUSD",
    interval="1m",
    market_type="um",
    data_type="klines",
    start_date="",                    # może być stringiem lub pustym
    end_date="2030-01-01 00:00:00",   # może być stringiem lub pustym
    split=False,
    delay=LAST_DATA_POINT_DELAY,
):
    # 1) Parsowanie start_date / end_date do obiektów date
    if start_date:
        start_dt = pd.to_datetime(start_date).date()
    else:
        start_dt = None

    if end_date:
        end_dt = pd.to_datetime(end_date).date()
    else:
        end_dt = date.today()

    # 2) przygotowanie URL i folderu
    if market_type in ("um", "cm"):
        base_url = (
            f"https://data.binance.vision/data/futures/{market_type}/monthly/"
            f"{data_type}/{ticker}/{interval}/{ticker}-{interval}-"
        )
        output_folder = os.path.join(
            DATA_DIR,
            f"binance_vision/futures_{market_type}/{data_type}/{ticker}{interval}"
        )
    elif market_type == "spot":
        base_url = (
            f"https://data.binance.vision/data/spot/monthly/"
            f"{data_type}/{ticker}/{interval}/{ticker}-{interval}-"
        )
        output_folder = os.path.join(
            DATA_DIR,
            f"binance_vision/spot/{data_type}/{ticker}{interval}"
        )
    else:
        raise ValueError("market_type must be one of 'um','cm','spot'")

    # 3) sprawdzenie cache
    csv_path = output_folder + ".csv"
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        df["Opened"] = pd.to_datetime(df["Opened"])
        # czy cache jest „świeży”?
        last_ts = int(df.iloc[-1]["Opened"].timestamp())
        if (time() - last_ts) <= delay:
            # nie trzeba pobierać nic nowego, tylko filtrowanie
            mask = df["Opened"].dt.date <= end_dt
            if start_dt:
                mask &= df["Opened"].dt.date >= start_dt
            df = df.loc[mask]
            if split:
                return df["Opened"], df.drop("Opened", axis=1)
            return df

    # 4) jeśli nie ma cache lub jest przeterminowany — pobieramy od zera
    # 4a) część miesięczna
    monthly_start = start_dt or date(2017, 1, 1)
    print(f"Collecting monthly from {monthly_start} to …")
    monthly_frames = _collect_to_date(
        base_url, output_folder,
        start_date=monthly_start,
        delta_itv="months"
    )

    # 4b) część dzienna — od pierwszego dnia obecnego miesiąca LUB od start_dt, jeśli późniejszy
    today = date.today()
    first_of_month = today.replace(day=1)
    daily_start = start_dt if (start_dt and start_dt > first_of_month) else first_of_month

    daily_url = base_url.replace("/monthly/", "/daily/")
    print(f"Collecting daily from {daily_start} to …")
    daily_frames = _collect_to_date(
        daily_url, output_folder,
        start_date=daily_start,
        delta_itv="days"
    )

    # 5) łączymy, wypełniamy i zapisujemy cache
    all_df = pd.concat(monthly_frames + daily_frames, ignore_index=True)
    fixed_df = _fix_and_fill_df(all_df, interval)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    fixed_df.to_csv(csv_path, index=False)

    # 6) ostateczne filtrowanie względem start_dt / end_dt
    mask = fixed_df["Opened"].dt.date <= end_dt
    if start_dt:
        mask &= fixed_df["Opened"].dt.date >= start_dt
    fixed_df = fixed_df.loc[mask]

    if split:
        return fixed_df["Opened"], fixed_df.drop("Opened", axis=1)
    return fixed_df

# Generate dicts

In [4]:
os.makedirs("data/ta_maps", exist_ok=True)

for pair in PAIRS:
    for itv in INTERVALS:
        print(f"Pobieram {pair} {itv} od 2024-04-01 …")
        df = by_BinanceVision(
            ticker=pair,
            interval=itv,
            market_type="um",
            start_date=MIN_DATE,
            end_date=""  # do dziś
        )
        # ustawiamy indeks na datetime
        df.set_index("Opened", inplace=True)

        # ----------------------------
        # tutaj zaczyna się blok z TA-Lib
        # ----------------------------
        ta_df = pd.DataFrame(index=df.index)
        ta_df['ADX']       = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)
        ta_df['APO']       = talib.APO(df['Close'], fastperiod=12, slowperiod=26, matype=0)
        ta_df['AROONOSC']  = talib.AROONOSC(df['High'], df['Low'], timeperiod=14)
        ta_df['BOP']       = talib.BOP(df['Open'], df['High'], df['Low'], df['Close'])
        ta_df['CCI']       = talib.CCI(df['High'], df['Low'], df['Close'], timeperiod=14)
        ta_df['CMO']       = talib.CMO(df['Close'], timeperiod=14)
        ta_df['MFI']       = talib.MFI(df['High'], df['Low'], df['Close'], df['Volume'], timeperiod=14)
        ta_df['MOM']       = talib.MOM(df['Close'], timeperiod=10)
        ta_df['ROCP']      = talib.ROCP(df['Close'], timeperiod=10)
        ta_df['RSI']       = talib.RSI(df['Close'], timeperiod=14)

        # STOCHF zwraca dwa wektory: fastk i fastd
        sto_k, sto_d      = talib.STOCHF(df['High'], df['Low'], df['Close'], fastk_period=5, fastd_period=3)
        ta_df['STOCHF']    = sto_k

        # STOCHRSI również: zwraca fastk i fastd — używamy fastk jako wartości oscylatora
        strsi_k, strsi_d  = talib.STOCHRSI(df['Close'], timeperiod=14, fastk_period=5, fastd_period=3)
        ta_df['STOCHRSI']  = strsi_k

        ta_df['ULTOSC']    = talib.ULTOSC(df['High'], df['Low'], df['Close'],
                                          timeperiod1=7, timeperiod2=14, timeperiod3=28)
        ta_df['WILLR']     = talib.WILLR(df['High'], df['Low'], df['Close'], timeperiod=14)
        ta_df['ADOSC']     = talib.ADOSC(df['High'], df['Low'], df['Close'], df['Volume'],
                                         fastperiod=3, slowperiod=10)
        ta_df['NATR']      = talib.NATR(df['High'], df['Low'], df['Close'], timeperiod=14)
        # ----------------------------
        # koniec bloku TA-Lib
        # ----------------------------

        # reset index, zostawiamy tylko Opened + kolumny wskaźników
        out = ta_df.reset_index()
        fname = f"data/ta_maps/{pair}{itv}.csv"
        out.to_csv(fname, index=False)
        print(f" → zapisano: {fname}")

Pobieram SOLUSDT 1m od 2024-04-01 …
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-26.zip
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-25.zip
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-24.zip
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-23.zip
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-22.zip
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-21.zip
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/1m/SOLUSDT-1m-2025-09-20.zip
 → zapisano: data/ta_maps/SOLUSDT1m.csv
Pobieram SOLUSDT 15m od 2024-04-01 …
downloading... https://data.binance.vision/data/futures/um/daily/klines/SOLUSDT/15m/SOLUSDT-15m-2025-09-26.zip
downloading..

# Old code

In [5]:
# for pair in FUTURES_PAIRS:
#     _ = by_BinanceVision(
#         ticker=pair,
#         interval="4h",
#         market_type="um",
#         data_type="klines",
#         start_date="2024-04-22 19:57:30",
#         end_date="2030-01-01 00:00:00",
#         split=False,
#         delay=LAST_DATA_POINT_DELAY)

In [6]:
# FUTURES_PAIRS = [
#     '1000000MOGUSDT',
#     '1000BONKUSDC',
#     '1000BONKUSDT',
#     '1000BTTCUSDT',
#     '1000CATUSDT',
#     '1000CHEEMSUSDT',
#     '1000FLOKIUSDT',
#     '1000LUNCBUSD',
#     '1000LUNCUSDT',
#     '1000PEPEUSDC',
#     '1000PEPEUSDT',
#     '1000RATSUSDT',
#     '1000SATSUSDT',
#     '1000SHIBBUSD',
#     '1000SHIBUSDC',
#     '1000SHIBUSDT',
#     '1000WHYUSDT',
#     '1000XECUSDT',
#     '1000XUSDT',
#     '1INCHUSDT',
#     '1MBABYDOGEUSDT',
#     'AAVEUSDT',
#     'ACEUSDT',
#     'ACHUSDT',
#     'ACTUSDT',
#     'ACXUSDT',
#     'ADABUSD',
#     'ADAUSDC',
#     'ADAUSDT',
#     'AERGOUSDT',
#     'AEROUSDT',
#     'AEVOUSDT',
#     'AGIXBUSD',
#     'AGIXUSDT',
#     'AGLDUSDT',
#     'AI16ZUSDT',
#     'AIUSDT',
#     'AIXBTUSDT',
#     'AKROUSDT',
#     'AKTUSDT',
#     'ALCHUSDT',
#     'ALGOUSDT',
#     'ALICEUSDT',
#     'ALPACAUSDT',
#     'ALPHAUSDT',
#     'ALTUSDT',
#     'AMBBUSD',
#     'AMBUSDT',
#     'ANCBUSD',
#     'ANCUSDT',
#     'ANIMEUSDT',
#     'ANKRUSDT',
#     'ANTUSDT',
#     'APEBUSD',
#     'APEUSDT',
#     'API3USDT',
#     'APTBUSD',
#     'APTUSDT',
#     'ARBUSDC',
#     'ARBUSDT',
#     'ARCUSDT',
#     'ARKMUSDT',
#     'ARKUSDT',
#     'ARPAUSDT',
#     'ARUSDT',
#     'ASTRUSDT',
#     'ATAUSDT',
#     'ATOMUSDT',
#     'AUCTIONBUSD',
#     'AUCTIONUSDT',
#     'AUDIOUSDT',
#     'AVAAIUSDT',
#     'AVAUSDT',
#     'AVAXBUSD',
#     'AVAXUSDC',
#     'AVAXUSDT',
#     'AXLUSDT',
#     'AXSUSDT',
#     'B3USDT',
#     'BADGERUSDT',
#     'BAKEUSDT',
#     'BALUSDT',
#     'BANANAS31USDT',
#     'BANANAUSDT',
#     'BANDUSDT',
#     'BANUSDT',
#     'BATUSDT',
#     'BBUSDT',
#     'BCHUSDC',
#     'BCHUSDT',
#     'BEAMXUSDT',
#     'BELUSDT',
#     'BERAUSDT',
#     'BICOUSDT',
#     'BIDUSDT',
#     'BIGTIMEUSDT',
#     'BIOUSDT',
#     'BLUEBIRDUSDT',
#     'BLURUSDT',
#     'BLZUSDT',
#     'BMTUSDT',
#     'BNBBUSD',
#     'BNBUSDC',
#     'BNBUSDT',
#     'BNTUSDT',
#     'BNXUSDT',
#     'BNXUSDTSETTLED',
#     'BOMEUSDC',
#     'BOMEUSDT',
#     'BONDUSDT',
#     'BRETTUSDT',
#     'BROCCOLI714USDT',
#     'BROCCOLIF3BUSDT',
#     'BRUSDT',
#     'BSVUSDT',
#     'BSWUSDT',
#     'BTCBUSD',
#     'BTCBUSD_210129',
#     'BTCBUSD_210226',
#     'BTCDOMUSDT',
#     'BTCSTUSDT',
#     'BTCUSDC',
#     'BTCUSDT',
#     'BTCUSDT_210326',
#     'BTCUSDT_210625',
#     'BTCUSDT_210924',
#     'BTCUSDT_211231',
#     'BTCUSDT_220325',
#     'BTCUSDT_220624',
#     'BTCUSDT_220930',
#     'BTCUSDT_221230',
#     'BTCUSDT_230331',
#     'BTCUSDT_230630',
#     'BTCUSDT_230929',
#     'BTCUSDT_231229',
#     'BTCUSDT_240329',
#     'BTCUSDT_240628',
#     'BTCUSDT_240927',
#     'BTCUSDT_241227',
#     'BTCUSDT_250328',
#     'BTCUSDT_250627',
#     'BTCUSDT_250926',
#     'BTSUSDT',
#     'BTTUSDT',
#     'BZRXUSDT',
#     'C98USDT',
#     'CAKEUSDT',
#     'CATIUSDT',
#     'CELOUSDT',
#     'CELRUSDT',
#     'CETUSUSDT',
#     'CFXUSDT',
#     'CGPTUSDT',
#     'CHESSUSDT',
#     'CHILLGUYUSDT',
#     'CHRUSDT',
#     'CHZUSDT',
#     'CKBUSDT',
#     'COCOSUSDT',
#     'COMBOUSDT',
#     'COMPUSDT',
#     'COOKIEUSDT',
#     'COSUSDT',
#     'COTIUSDT',
#     'COWUSDT',
#     'CRVUSDC',
#     'CRVUSDT',
#     'CTKUSDT',
#     'CTSIUSDT',
#     'CVCUSDT',
#     'CVXBUSD',
#     'CVXUSDT',
#     'CYBERUSDT',
#     'DARUSDT',
#     'DASHUSDT',
#     'DEFIUSDT',
#     'DEGENUSDT',
#     'DEGOUSDT',
#     'DENTUSDT',
#     'DEXEUSDT',
#     'DFUSDT',
#     'DGBUSDT',
#     'DIAUSDT',
#     'DODOBUSD',
#     'DODOUSDT',
#     'DODOXUSDT',
#     'DOGEBUSD',
#     'DOGEUSDC',
#     'DOGEUSDT',
#     'DOGSUSDT',
#     'DOTBUSD',
#     'DOTECOUSDT',
#     'DOTUSDT',
#     'DRIFTUSDT',
#     'DUSDT',
#     'DUSKUSDT',
#     'DYDXUSDT',
#     'DYMUSDT',
#     'EDUUSDT',
#     'EGLDUSDT',
#     'EIGENUSDT',
#     'ENAUSDC',
#     'ENAUSDT',
#     'ENJUSDT',
#     'ENSUSDT',
#     'EOSUSDT',
#     'EPICUSDT',
#     'ETCBUSD',
#     'ETCUSDT',
#     'ETHBTC',
#     'ETHBUSD',
#     'ETHFIUSDC',
#     'ETHFIUSDT',
#     'ETHUSDC',
#     'ETHUSDT',
#     'ETHUSDT_210326',
#     'ETHUSDT_210625',
#     'ETHUSDT_210924',
#     'ETHUSDT_211231',
#     'ETHUSDT_220325',
#     'ETHUSDT_220624',
#     'ETHUSDT_220930',
#     'ETHUSDT_221230',
#     'ETHUSDT_230331',
#     'ETHUSDT_230630',
#     'ETHUSDT_230929',
#     'ETHUSDT_231229',
#     'ETHUSDT_240329',
#     'ETHUSDT_240628',
#     'ETHUSDT_240927',
#     'ETHUSDT_241227',
#     'ETHUSDT_250328',
#     'ETHUSDT_250627',
#     'ETHUSDT_250926',
#     'ETHWUSDT',
#     'FARTCOINUSDT',
#     'FETUSDT',
#     'FIDAUSDT',
#     'FILBUSD',
#     'FILUSDC',
#     'FILUSDT',
#     'FIOUSDT',
#     'FLMUSDT',
#     'FLOWUSDT',
#     'FLUXUSDT',
#     'FOOTBALLUSDT',
#     'FORMUSDT',
#     'FRONTUSDT',
#     'FTMBUSD',
#     'FTMUSDT',
#     'FTTBUSD',
#     'FTTUSDT',
#     'FUNUSDT',
#     'FXSUSDT',
#     'GALABUSD',
#     'GALAUSDT',
#     'GALBUSD',
#     'GALUSDT',
#     'GASUSDT',
#     'GHSTUSDT',
#     'GLMRUSDT',
#     'GLMUSDT',
#     'GMTBUSD',
#     'GMTUSDT',
#     'GMXUSDT',
#     'GOATUSDT',
#     'GPSUSDT',
#     'GRASSUSDT',
#     'GRIFFAINUSDT',
#     'GRTUSDT',
#     'GTCUSDT',
#     'GUNUSDT',
#     'GUSDT',
#     'HBARUSDC',
#     'HBARUSDT',
#     'HEIUSDT',
#     'HFTUSDT',
#     'HIFIUSDT',
#     'HIGHUSDT',
#     'HIPPOUSDT',
#     'HIVEUSDT',
#     'HMSTRUSDT',
#     'HNTUSDT',
#     'HOOKUSDT',
#     'HOTUSDT',
#     'ICPBUSD',
#     'ICPUSDT',
#     'ICPUSDT_SETTLED',
#     'ICXUSDT',
#     'IDEXUSDT',
#     'IDUSDT',
#     'ILVUSDT',
#     'IMXUSDT',
#     'INJUSDT',
#     'IOSTUSDT',
#     'IOTAUSDT',
#     'IOTXUSDT',
#     'IOUSDT',
#     'IPUSDC',
#     'IPUSDT',
#     'JASMYUSDT',
#     'JELLYJELLYUSDT',
#     'JOEUSDT',
#     'JTOUSDT',
#     'JUPUSDT',
#     'KAIAUSDT',
#     'KAITOUSDC',
#     'KAITOUSDT',
#     'KASUSDT',
#     'KAVAUSDT',
#     'KDAUSDT',
#     'KEEPUSDT',
#     'KEYUSDT',
#     'KLAYUSDT',
#     'KMNOUSDT',
#     'KNCUSDT',
#     'KOMAUSDT',
#     'KSMUSDT',
#     'LAYERUSDT',
#     'LDOBUSD',
#     'LDOUSDT',
#     'LENDUSDT',
#     'LEVERBUSD',
#     'LEVERUSDT',
#     'LINAUSDT',
#     'LINKBUSD',
#     'LINKUSDC',
#     'LINKUSDT',
#     'LISTAUSDT',
#     'LITUSDT',
#     'LOKAUSDT',
#     'LOOMUSDT',
#     'LPTUSDT',
#     'LQTYUSDT',
#     'LRCUSDT',
#     'LSKUSDT',
#     'LTCBUSD',
#     'LTCUSDC',
#     'LTCUSDT',
#     'LUMIAUSDT',
#     'LUNA2BUSD',
#     'LUNA2USDT',
#     'LUNABUSD',
#     'LUNAUSDT',
#     'MAGICUSDT',
#     'MANAUSDT',
#     'MANTAUSDT',
#     'MASKUSDT',
#     'MATICBUSD',
#     'MATICUSDC',
#     'MATICUSDT',
#     'MAVIAUSDT',
#     'MAVIAUSDTSETTLED',
#     'MAVUSDT',
#     'MBLUSDT',
#     'MBOXUSDT',
#     'MDTUSDT',
#     'MELANIAUSDT',
#     'MEMEUSDT',
#     'METISUSDT',
#     'MEUSDT',
#     'MEWUSDT',
#     'MINAUSDT',
#     'MINAUSDTSETTLED',
#     'MKRUSDT',
#     'MLNUSDT',
#     'MOCAUSDT',
#     'MOODENGUSDT',
#     'MORPHOUSDT',
#     'MOVEUSDT',
#     'MOVRUSDT',
#     'MTLUSDT',
#     'MUBARAKUSDT',
#     'MYROUSDT',
#     'NEARBUSD',
#     'NEARUSDC',
#     'NEARUSDT',
#     'NEIROETHUSDT',
#     'NEIROUSDT',
#     'NEOUSDC',
#     'NEOUSDT',
#     'NFPUSDT',
#     'NILUSDT',
#     'NKNUSDT',
#     'NMRUSDT',
#     'NOTUSDT',
#     'NTRNUSDT',
#     'NULSUSDT',
#     'NUUSDT',
#     'OCEANUSDT',
#     'OGNUSDT',
#     'OMGUSDT',
#     'OMNIUSDT',
#     'OMUSDT',
#     'ONDOUSDT',
#     'ONEUSDT',
#     'ONGUSDT',
#     'ONTUSDT',
#     'OPUSDT',
#     'ORBSUSDT',
#     'ORCAUSDT',  
#     'ORDIUSDC',  
#     'ORDIUSDT',  
#     'OXTUSDT',  
#     'PARTIUSDT',  
#     'PAXGUSDT',  
#     'PENDLEUSDT',  
#     'PENGUUSDT',  
#     'PEOPLEUSDT',  
#     'PERPUSDT',  
#     'PHAUSDT',  
#     'PHBBUSD',  
#     'PHBUSDT',  
#     'PIPPINUSDT',  
#     'PIXELUSDT',  
#     'PLUMEUSDT',  
#     'PNUTUSDC',  
#     'PNUTUSDT',  
#     'POLUSDT',  
#     'POLYXUSDT',  
#     'PONKEUSDT',  
#     'POPCATUSDT',  
#     'PORTALUSDT',  
#     'POWRUSDT',  
#     'PROMUSDT',  
#     'PYTHUSDT',  
#     'QNTUSDT',  
#     'QTUMUSDT',  
#     'QUICKUSDT',  
#     'RADUSDT',  
#     'RAREUSDT',  
#     'RAYSOLUSDT',  
#     'RAYUSDT',  
#     'RDNTUSDT',  
#     'REDUSDT',  
#     'REEFUSDT',  
#     'REIUSDT',  
#     'RENDERUSDT',  
#     'RENUSDT',  
#     'REZUSDT',  
#     'RIFUSDT',  
#     'RLCUSDT',  
#     'RNDRUSDT',  
#     'RONINUSDT',  
#     'ROSEUSDT',  
#     'RPLUSDT',  
#     'RSRUSDT',  
#     'RUNEUSDT',  
#     'RVNUSDT',  
#     'SAFEUSDT',  
#     'SAGAUSDT',  
#     'SANDBUSD',  
#     'SANDUSDT',  
#     'SANTOSUSDT',  
#     'SCRTUSDT',  
#     'SCRUSDT',  
#     'SCUSDT',  
#     'SEIUSDT',  
#     'SFPUSDT',  
#     'SHELLUSDT',  
#     'SIRENUSDT',  
#     'SKLUSDT',  
#     'SLERFUSDT',  
#     'SLPUSDT',  
#     'SNTUSDT',  
#     'SNXUSDT',  
#     'SOLBUSD',  
#     'SOLUSDC',  
#     'SOLUSDT',  
#     'SOLVUSDT',  
#     'SONICUSDT',  
#     'SPELLUSDT',  
#     'SPXUSDT',  
#     'SRMUSDT',  
#     'SSVUSDT',  
#     'STEEMUSDT',  
#     'STGUSDT',  
#     'STMXUSDT',  
#     'STORJUSDT',  
#     'STPTUSDT',  
#     'STRAXUSDT',  
#     'STRKUSDT',  
#     'STXUSDT',  
#     'SUIUSDC',  
#     'SUIUSDT',  
#     'SUNUSDT',  
#     'SUPERUSDT',  
#     'SUSDT',  
#     'SUSHIUSDT',  
#     'SWARMSUSDT',  
#     'SWELLUSDT',  
#     'SXPUSDT',  
#     'SYNUSDT',  
#     'SYSUSDT',  
#     'TAOUSDT',  
#     'THETAUSDT',  
#     'THEUSDT',  
#     'TIAUSDC',  
#     'TIAUSDT',  
#     'TLMBUSD',  
#     'TLMUSDT',  
#     'TLMUSDTSETTLED',  
#     'TNSRUSDT',  
#     'TOKENUSDT',  
#     'TOMOUSDT',  
#     'TONUSDT',  
#     'TRBUSDT',  
#     'TROYUSDT',  
#     'TRUMPUSDC',  
#     'TRUMPUSDT',  
#     'TRUUSDT',  
#     'TRXBUSD',  
#     'TRXUSDT',  
#     'TSTUSDT',  
#     'TURBOUSDT',  
#     'TUSDT',  
#     'TUTUSDT',  
#     'TWTUSDT',  
#     'UMAUSDT',  
#     'UNFIUSDT',  
#     'UNIBUSD',  
#     'UNIUSDT',  
#     'USDCUSDT',  
#     'USTCUSDT',  
#     'USUALUSDT',  
#     'UXLINKUSDT',  
#     'VANAUSDT',  
#     'VANRYUSDT',  
#     'VELODROMEUSDT',  
#     'VETUSDT',  
#     'VICUSDT',  
#     'VIDTUSDT',  
#     'VINEUSDT',  
#     'VIRTUALUSDT',  
#     'VOXELUSDT',  
#     'VTHOUSDT',  
#     'VVVUSDT',  
#     'WALUSDT',  
#     'WAVESBUSD',  
#     'WAVESUSDT',  
#     'WAXPUSDT',  
#     'WIFUSDC',  
#     'WIFUSDT',  
#     'WLDUSDC',  
#     'WLDUSDT',  
#     'WOOUSDT',  
#     'WUSDT',  
#     'XAIUSDT',  
#     'XEMUSDT',  
#     'XLMUSDT',  
#     'XMRUSDT',  
#     'XRPBUSD',  
#     'XRPUSDC',  
#     'XRPUSDT',  
#     'XTZUSDT',  
#     'XVGUSDT',  
#     'XVSUSDT',  
#     'YFIIUSDT',  
#     'YFIUSDT',  
#     'YGGUSDT',  
#     'ZECUSDT',  
#     'ZENUSDT',  
#     'ZEREBROUSDT',  
#     'ZETAUSDT',  
#     'ZILUSDT',  
#     'ZKUSDT',  
#     'ZROUSDT',  
#     'ZRXUSDT',  
# ]

In [7]:
# import os
# import glob
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

# # --- KONFIGURACJA ŚCIEŻEK ---
# data_path = r'C:\Cloud\filips19mail\github\solanaDEXtokenCollector\data\binance_vision\futures_um\klines'
# all_files = glob.glob(os.path.join(data_path, '*.csv'))
# reference_filename = 'SOLUSDT4h.csv'
# reference_filepath = os.path.join(data_path, reference_filename)
# date_cutoff = '2024-04-01'

# # --- FUNKCJA ŁADUJĄCA I PRZYGOTOWUJĄCA DANE ---
# def load_and_prepare(path):
#     df = pd.read_csv(path, parse_dates=['Opened'])
#     cols = ['Open','High','Low','Close','Volume']
#     df[cols] = df[cols].astype(float)
#     df = df[df['Opened'] >= date_cutoff].copy()
#     df['hl2']       = (df['High'] + df['Low']) / 2
#     df['close_pct'] = df['Close'].pct_change()
#     df['vol_pct']   = df['Volume'].pct_change()
#     df = df.dropna(subset=['hl2','close_pct','vol_pct'])
#     return df.set_index('Opened')

# # --- WCZYTANIE SERII REFERENCYJNEJ ---
# ref_df = load_and_prepare(reference_filepath)

# # --- OBLICZENIE KORELACJI DLA KAŻDEJ CECHY W JEDNYM SŁOWNIKU ---
# features = ['hl2', 'close_pct', 'vol_pct']
# correlations = {feat: {} for feat in features}

# for fp in all_files:
#     name = os.path.basename(fp)
#     if name == reference_filename:
#         continue

#     # sprawdź, czy mamy dane przed / w dniu cutoff
#     raw = pd.read_csv(fp, parse_dates=['Opened'])
#     if raw['Opened'].min() > pd.to_datetime(date_cutoff):
#         print(f"Pomijam {name}: brak danych przed {date_cutoff}")
#         continue

#     df = load_and_prepare(fp)
#     common_idx = ref_df.index.intersection(df.index)
#     if len(common_idx) < 10:
#         continue

#     for feat in features:
#         correlations[feat][name] = ref_df.loc[common_idx, feat].corr(df.loc[common_idx, feat])

# # --- WYŚWIETLENIE I RYSOWANIE ---
# for feat in features:
#     # teraz to działa, bo correlations[feat] jest wypełnione
#     sorted_items = sorted(correlations[feat].items(), key=lambda x: -abs(x[1]))

#     print(f"\n=== Top 25 dla cechy '{feat}' ===")
#     for fname, corr in sorted_items[:25]:
#         print(f"{fname:25s} -> {corr:.3f}")

#     # wykres słupkowy Top 25
#     top25 = sorted_items[:25]
#     names  = [n for n,_ in top25]
#     vals   = [v for _,v in top25]

#     plt.figure(figsize=(12,6))
#     plt.bar(names, vals)
#     plt.xticks(rotation=90)
#     plt.title(f"Top 25 korelacji dla '{feat}'")
#     plt.xlabel("Para/plik")
#     plt.ylabel("Korelacja")
#     plt.tight_layout()
#     plt.show()
